Leer los archivos csv

In [ ]:
import csv
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import json


ruta_movies = Path(r'DataSets\\tmdb_5000_movies.csv')
df_movies = pd.read_csv(ruta_movies)
print('Encabezado y primeras 5 filas:')
print(df_movies.head())
print('\nDescripcion general:')
print(df_movies.describe())
print('\nInformacion de columnas:')
print(df_movies.info())

print("Nulos en df_movies:")
print(df_movies.isnull().sum())

#with ruta_movies.open('r', newline='', encoding='utf-8') as f:
    #reader = csv.reader(f)
    #encabezado = next(reader)
    #print('Encabezado: ', encabezado)
    #for fila in reader:
        #print('Fila', fila)

In [ ]:
ruta_credits = Path(r'DataSets\\tmdb_5000_credits.csv')

df_credits = pd.read_csv(ruta_credits)
print('\nEncabezado y primeras 5 filas de creditos:')
print(df_credits.head())

print("\nNulos en df_credits:")
print(df_credits.isnull().sum())


#with ruta_credits.open('r', newline='', #encoding='utf-8') as f:
    #reader = csv.reader(f)
    #encabezado = next(reader)
    #print('Encabezado: ', encabezado)
    #for fila in reader:
        #print('Fila', fila)

In [ ]:
print("\nNulos en df_credits:")
print(df_credits.isnull().sum())

print("Nulos en df_movies:")
print(df_movies.isnull().sum())

In [ ]:
#Como solo el csv de peliculas tiene nulos, no encargaremos de aquellos y crearemos un nuevo csv sin errores.
df_movies = df_movies.drop(columns=['homepage', 'tagline'], errors='ignore')  # errors='ignore' por si alguna no existe
df_movies = df_movies[df_movies['budget'] > 0].copy()

# Verificar las columnas restantes
print("Columnas después de eliminación:")
print(df_movies.columns)

# Guardar el DataFrame modificado en un nuevo CSV
ruta_nuevo = Path('nuevosDatasets/tmdb_5000_movies_limpia.csv')
df_movies.to_csv(ruta_nuevo, index=False)  # index=False para no incluir el índice como columna

print(f"Archivo guardado en: {ruta_nuevo}")


#Se remueven
df_movies['release_date'] = pd.to_datetime(df_movies['release_date'], errors='coerce')

# Eliminar filas donde release_date es null (NaT)
df_movies = df_movies.dropna(subset=['release_date'])

# Eliminar filas donde runtime es null (NaN)
df_movies = df_movies.dropna(subset=['runtime'])

# Imputar null en overview con "Sin overview"
df_movies['overview'] = df_movies['overview'].fillna("Sin overview")

# Verificar nulos restantes en estas columnas
print("Nulos después del tratamiento:")
print(df_movies[['release_date', 'runtime', 'overview']].isnull().sum())

# Verificar el número de películas restantes
print(f"Número de películas después de eliminación: {len(df_movies)}")

# Guardar el DataFrame actualizado en el CSV
df_movies.to_csv(ruta_nuevo, index=False)

print(f"Archivo actualizado guardado en: {ruta_nuevo}")



In [ ]:
# Resumen inicial de df_movies
print("Resumen de df_movies:")
print(df_movies.describe())  # Estadísticas numéricas (media, mediana, etc.)
print(df_movies.info())  # Tipos y nulos
print(df_movies.head())  # Primeras filas

# Resumen inicial de df_credits
print("\nResumen de df_credits:")
print(df_credits.describe())
print(df_credits.info())
print(df_credits.head())

### Eje 1: Rentabilidad (ROI) por género o país
# ROI = revenue / budget (evitar divisiones por 0)
df_movies['ROI'] = np.where(df_movies['budget'] > 0, df_movies['revenue'] / df_movies['budget'], np.nan)

# Parsear 'genres' de string JSON a lista real
import ast  # Para parsear strings JSON-like
df_movies['genres_list'] = df_movies['genres'].apply(lambda x: [g['name'] for g in ast.literal_eval(x)] if isinstance(x, str) else [])

# Explotar géneros para análisis
genres_exploded = df_movies.explode('genres_list')

# ROI promedio por género
roi_por_genero = genres_exploded.groupby('genres_list')['ROI'].mean().sort_values(ascending=False).head(10)

# Gráfico
plt.figure(figsize=(10, 6))
sns.barplot(x=roi_por_genero.values, y=roi_por_genero.index)
plt.title('Top 10 Géneros por ROI Promedio')
plt.xlabel('ROI Promedio')
plt.ylabel('Género')
plt.show()

# Conclusión: Los géneros como [ej: Horror] tienen alto ROI porque son baratos de producir y generan buena taquilla. Exportamos a CSV.
roi_por_genero.to_csv('nuevosDatasets/roi_por_genero.csv', index=True)


In [ ]:
### Eje 2: Relación entre presupuesto y rating
# Correlación Pearson
correlacion = df_movies['budget'].corr(df_movies['vote_average'])
print(f"Correlación Pearson entre budget y vote_average: {correlacion}")

# Gráfico de dispersión
plt.figure(figsize=(11, 6))
sns.scatterplot(data=df_movies, x='budget', y='vote_average', alpha=0.5, s=20)
plt.title('Relación Presupuesto vs Rating')
plt.xlabel('Presupuesto (budget)')
plt.ylabel('Rating Promedio (vote_average)')

marcas_bajas = np.arange(0, 100000001, 20000000)
marcas_altas = np.arange(175000000, 400000001, 75000000)
marcas_personalizadas = np.unique(np.concatenate([marcas_bajas, marcas_altas]))


ax = plt.gca()

ax.xaxis.set_major_locator(ticker.MultipleLocator(50000000))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(
    # La función lambda recibe el valor numérico (x) y la posición (pos)
    lambda x, pos: f'${int(x/1000000)}M'))

ax.set_xticks(marcas_personalizadas)

plt.show()

# Conclusión: Hay una correlación baja positiva ({correlacion}), indicando que más presupuesto no garantiza mejor rating, pero hay outliers con alto budget y rating.



In [ ]:
### Eje 3: Evolución de la duración de películas en los últimos 50 años
# Crear columna 'release_year' a partir de 'release_date' (asumiendo que ya es datetime de celdas anteriores)
df_movies['release_date'] = pd.to_datetime(df_movies['release_date'], errors='coerce')

# Crear columna 'release_year' a partir de 'release_date' (asumiendo que ya es datetime de celdas anteriores)
df_movies['release_year'] = df_movies['release_year'] = df_movies['release_date'].dt.year.astype('Int64')  # Extraer el año como entero, usando Int64 para manejar NaN


# Filtrar últimos 50 años (asumiendo current_date ~2025)
df_recent = df_movies[df_movies['release_year'] >= 1975]

# Duración promedio por década
df_recent['decada'] = (df_recent['release_year'] // 10) * 10
duracion_por_decada = df_recent.groupby('decada')['runtime'].median()

# Gráfico
plt.figure(figsize=(10, 6))
duracion_por_decada.plot(kind='line', marker='o')
plt.title('Evolución de la Duración Mediana de Películas por Década')
plt.xlabel('Década')
plt.ylabel('Duración Mediana (minutos)')
plt.show()

# Conclusión: La duración ha aumentado ligeramente en las últimas décadas, posiblemente por blockbusters más largos. Exportamos a JSON.
duracion_por_decada.to_json('nuevosDatasets/duracion_por_decada.json', orient='index')

### Conclusiones Generales del EDA
# - El dataset tiene ~4800 películas, con presupuestos variando de 0 a cientos de millones.
# - Patrones clave: [resumir hallazgos de los ejes].
# - Para la API, usaremos los archivos generados (e.g., roi_por_genero.csv) en endpoints como /top_generos.